# AIDA — Rare / Significant Cell Type Analysis

This notebook documents cell types that show large F1 changes across SG_Chinese proportions
but have very few cells in the query set:

| Cell type | Δ F1 (100% − 0%) | Support in query |
|---|---|---|
| innate lymphoid cell (ILC) | +0.132 | 16 |
| double negative T regulatory cell (dnT) | +0.140 | 4 |
| pre-conventional dendritic cell (pre-conv DC) | −0.035 | 24 |

Small support means per-replicate F1 is noisy; we check annotation quality
and summarise the F1 trend from the full-curve results.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
import scanpy as sc
import numpy as np
import pandas as pd
import json
import scipy.stats as stats

In [2]:
# ── Load AIDA in backed mode + fix query donors (seed=42) ──────────────────
adata = sc.read_h5ad('AIDA.h5ad', backed='r')

sg_chinese_donors = adata.obs[
    adata.obs['self_reported_ethnicity'] == 'Singaporean Chinese'
]['donor_id'].unique()

np.random.seed(42)
query_donors = np.random.choice(sg_chinese_donors, size=20, replace=False)

print(f"Query donors: {len(query_donors)}")
query_mask = adata.obs['donor_id'].isin(query_donors)

Query donors: 20


## 1. Innate lymphoid cell (ILC)

ILC shows the largest positive F1 gain among all cell types: **+0.132**.
However, only 16 ILC cells are in the query set, so we inspect their
Annotation_Level4 labels to understand the subtype context.

In [3]:
# ── ILC annotation quality in query ───────────────────────────────────────
ilc = adata.obs[query_mask & (adata.obs['cell_type'] == 'innate lymphoid cell')]
print(f"Total ILC cells in query: {len(ilc)}")
print()
print(ilc['Annotation_Level4'].value_counts().head(10))

Total ILC cells in query: 16

Annotation_Level4
ILC_XCL1hi            14
ILC_SCART1hi           2
B_unknown              0
CD4+_T_cm              0
CD4+_T_cm_IFNhi        0
CD4+_T_cyt             0
CD4+_T_em              0
CD4+_T_memory          0
CD4+_T_naive           0
CD4+_T_naive_IFNhi     0
Name: count, dtype: int64


In [4]:
# ── ILC F1 across proportions from all_reports.json ───────────────────────
with open('all_reports.json', 'r') as f:
    all_reports = json.load(f)

rows_ilc = []
for entry in all_reports:
    r = entry['report']
    if 'innate lymphoid cell' in r:
        rows_ilc.append({
            'proportion': entry['proportion'],
            'repeat':     entry['repeat'],
            'precision':  r['innate lymphoid cell']['precision'],
            'recall':     r['innate lymphoid cell']['recall'],
            'f1':         r['innate lymphoid cell']['f1-score']
        })

df_ilc = pd.DataFrame(rows_ilc)
summary_ilc = df_ilc.groupby('proportion')[['precision', 'recall', 'f1']].agg(['mean', 'std']).round(3)
print("ILC F1 by proportion:")
print(summary_ilc)

# Kendall τ
props     = summary_ilc.index.tolist()
f1vals    = summary_ilc['f1']['mean'].tolist()
tau, p    = stats.kendalltau(props, f1vals)
print(f"\nKendall τ: {tau:.3f}, p = {p:.4f}")
print(f"F1 range : {min(f1vals):.3f} → {max(f1vals):.3f}")
print(f"Δ F1 (100% − 0%): {f1vals[-1] - f1vals[0]:+.3f}")

ILC F1 by proportion:
           precision        recall            f1       
                mean    std   mean    std   mean    std
proportion                                             
0.0            0.915  0.094  0.413  0.091  0.563  0.093
0.1            0.901  0.107  0.439  0.122  0.581  0.114
0.2            0.887  0.091  0.469  0.078  0.608  0.069
0.3            0.954  0.075  0.410  0.090  0.567  0.092
0.4            0.928  0.074  0.451  0.091  0.601  0.084
0.5            0.822  0.118  0.462  0.081  0.583  0.071
0.6            0.916  0.074  0.495  0.067  0.638  0.056
0.7            0.837  0.118  0.505  0.086  0.623  0.081
0.8            0.815  0.106  0.496  0.066  0.614  0.070
0.9            0.837  0.068  0.534  0.046  0.651  0.049
1.0            0.858  0.066  0.589  0.059  0.695  0.044

Kendall τ: 0.709, p = 0.0016
F1 range : 0.563 → 0.695
Δ F1 (100% − 0%): +0.132


## 2. Double negative T regulatory cell (dnT)

dnT has the largest per-cell-type F1 gain: **+0.140**, but with only **4 cells** in the query
the F1 is extremely noisy (a single mis/correct call swings F1 by ~0.25).

In [5]:
# ── dnT annotation quality in query ───────────────────────────────────────
dntreg = adata.obs[query_mask & (adata.obs['cell_type'] == 'double negative T regulatory cell')]
print(f"Total dnT cells in query: {len(dntreg)}")
print()
print(dntreg['Annotation_Level4'].value_counts().head(10))

Total dnT cells in query: 4

Annotation_Level4
dnT                    4
B_unknown              0
CD4+_T_cm              0
CD4+_T_cm_IFNhi        0
CD4+_T_cyt             0
CD4+_T_em              0
CD4+_T_memory          0
CD4+_T_naive           0
CD4+_T_naive_IFNhi     0
CD4+_T_naive_SOX4hi    0
Name: count, dtype: int64


In [6]:
# ── dnT F1 across proportions ─────────────────────────────────────────────
rows_dnt = []
for entry in all_reports:
    r = entry['report']
    if 'double negative T regulatory cell' in r:
        rows_dnt.append({
            'proportion': entry['proportion'],
            'repeat':     entry['repeat'],
            'precision':  r['double negative T regulatory cell']['precision'],
            'recall':     r['double negative T regulatory cell']['recall'],
            'f1':         r['double negative T regulatory cell']['f1-score']
        })

df_dnt = pd.DataFrame(rows_dnt)
summary_dnt = df_dnt.groupby('proportion')[['precision', 'recall', 'f1']].agg(['mean', 'std']).round(3)
print("dnT F1 by proportion:")
print(summary_dnt)

props_d  = summary_dnt.index.tolist()
f1vals_d = summary_dnt['f1']['mean'].tolist()
tau_d, p_d = stats.kendalltau(props_d, f1vals_d)
print(f"\nKendall τ: {tau_d:.3f}, p = {p_d:.4f}")
print(f"F1 range : {min(f1vals_d):.3f} → {max(f1vals_d):.3f}")
print(f"Δ F1 (100% − 0%): {f1vals_d[-1] - f1vals_d[0]:+.3f}")
print()
print("Caution: only 4 cells in query. F1 variance per replicate is very high.")

dnT F1 by proportion:
           precision        recall            f1       
                mean    std   mean    std   mean    std
proportion                                             
0.0            0.954  0.117  0.569  0.165  0.698  0.151
0.1            0.938  0.130  0.618  0.127  0.738  0.115
0.2            0.930  0.117  0.729  0.125  0.808  0.090
0.3            0.991  0.056  0.694  0.224  0.795  0.174
0.4            0.970  0.087  0.750  0.134  0.839  0.100
0.5            0.919  0.139  0.764  0.103  0.829  0.096
0.6            0.958  0.094  0.743  0.042  0.834  0.050
0.7            0.878  0.162  0.736  0.083  0.790  0.089
0.8            0.933  0.112  0.793  0.142  0.844  0.084
0.9            0.874  0.134  0.793  0.096  0.827  0.096
1.0            0.955  0.098  0.758  0.117  0.838  0.088

Kendall τ: 0.491, p = 0.0405
F1 range : 0.698 → 0.844
Δ F1 (100% − 0%): +0.140

Caution: only 4 cells in query. F1 variance per replicate is very high.


## 3. Pre-conventional dendritic cell (pre-conv DC)
Pre-conv DC shows a Δ F1 of −0.035, but Kendall τ = 0.127 (p = 0.648) is **not statistically significant** — no monotonic trend is confirmed.
The small support also makes it noisy.

In [7]:
# ── pre-conv DC annotation quality in query ────────────────────────────────
predc = adata.obs[query_mask & (adata.obs['cell_type'] == 'pre-conventional dendritic cell')]
print(f"Total pre-conv DC cells in query: {len(predc)}")
if len(predc) > 0:
    print()
    print(predc['Annotation_Level4'].value_counts().head(10))

Total pre-conv DC cells in query: 24

Annotation_Level4
DC_SIGLEC6hi           24
B_unknown               0
CD4+_T_cm               0
CD4+_T_cm_IFNhi         0
CD4+_T_cyt              0
CD4+_T_em               0
CD4+_T_memory           0
CD4+_T_naive            0
CD4+_T_naive_IFNhi      0
CD4+_T_naive_SOX4hi     0
Name: count, dtype: int64


In [8]:
# ── pre-conv DC F1 across proportions ─────────────────────────────────────
rows_dc = []
for entry in all_reports:
    r = entry['report']
    if 'pre-conventional dendritic cell' in r:
        rows_dc.append({
            'proportion': entry['proportion'],
            'repeat':     entry['repeat'],
            'precision':  r['pre-conventional dendritic cell']['precision'],
            'recall':     r['pre-conventional dendritic cell']['recall'],
            'f1':         r['pre-conventional dendritic cell']['f1-score']
        })

df_dc = pd.DataFrame(rows_dc)
if len(df_dc) > 0:
    summary_dc = df_dc.groupby('proportion')[['precision', 'recall', 'f1']].agg(['mean', 'std']).round(3)
    print("pre-conv DC F1 by proportion:")
    print(summary_dc)

    props_dc  = summary_dc.index.tolist()
    f1vals_dc = summary_dc['f1']['mean'].tolist()
    tau_dc, p_dc = stats.kendalltau(props_dc, f1vals_dc)
    print(f"\nKendall τ: {tau_dc:.3f}, p = {p_dc:.4f}")
    print(f"F1 range : {min(f1vals_dc):.3f} → {max(f1vals_dc):.3f}")
    print(f"Δ F1 (100% − 0%): {f1vals_dc[-1] - f1vals_dc[0]:+.3f}")
else:
    print("No pre-conv DC entries in all_reports.json.")

pre-conv DC F1 by proportion:
           precision        recall            f1       
                mean    std   mean    std   mean    std
proportion                                             
0.0            0.875  0.079  0.605  0.104  0.708  0.071
0.1            0.804  0.077  0.538  0.077  0.642  0.069
0.2            0.798  0.092  0.568  0.100  0.658  0.082
0.3            0.808  0.079  0.595  0.102  0.678  0.070
0.4            0.782  0.099  0.556  0.098  0.646  0.088
0.5            0.787  0.082  0.587  0.087  0.667  0.065
0.6            0.752  0.071  0.591  0.118  0.653  0.078
0.7            0.727  0.073  0.595  0.074  0.652  0.060
0.8            0.750  0.076  0.618  0.070  0.676  0.063
0.9            0.757  0.051  0.639  0.059  0.692  0.046
1.0            0.778  0.042  0.595  0.045  0.673  0.038

Kendall τ: 0.127, p = 0.6481
F1 range : 0.642 → 0.708
Δ F1 (100% − 0%): -0.035


## 4. Broad scan: all cell types with large |Δ F1|

We scan all cell types in `all_reports.json` and rank by absolute change in mean F1
between 0 % and 100 % SG_Chinese reference.

In [9]:
# ── Broad scan across all reported cell types ──────────────────────────────
from collections import defaultdict

cell_data = defaultdict(list)
for entry in all_reports:
    prop = entry['proportion']
    for ct, metrics in entry['report'].items():
        if isinstance(metrics, dict) and 'f1-score' in metrics:
            cell_data[ct].append({
                'proportion': prop,
                'f1': metrics['f1-score'],
                'support': metrics.get('support', None)
            })

rows_scan = []
for ct, entries in cell_data.items():
    df_ct = pd.DataFrame(entries)
    mean_by_prop = df_ct.groupby('proportion')['f1'].mean()
    if 0.0 in mean_by_prop.index and 1.0 in mean_by_prop.index:
        delta   = mean_by_prop[1.0] - mean_by_prop[0.0]
        f1_0    = mean_by_prop[0.0]
        f1_100  = mean_by_prop[1.0]
        support = df_ct['support'].median()
        tau, p  = stats.kendalltau(df_ct['proportion'], df_ct['f1'])
        rows_scan.append({
            'cell_type': ct,
            'f1_at_0pct':   round(f1_0, 3),
            'f1_at_100pct': round(f1_100, 3),
            'delta_f1':     round(delta, 3),
            'abs_delta':    round(abs(delta), 3),
            'tau':          round(tau, 3),
            'p_tau':        round(p, 4),
            'median_support': support
        })

df_scan = pd.DataFrame(rows_scan).sort_values('abs_delta', ascending=False)
print("Top 20 cell types by |Δ F1|:")
print(df_scan.head(20).to_string(index=False))

Top 20 cell types by |Δ F1|:
                                            cell_type  f1_at_0pct  f1_at_100pct  delta_f1  abs_delta    tau  p_tau  median_support
                    double negative T regulatory cell       0.698         0.838     0.140      0.140  0.179 0.0000             4.0
                                 innate lymphoid cell       0.563         0.695     0.132      0.132  0.271 0.0000            16.0
                                  natural killer cell       0.601         0.675     0.074      0.074  0.231 0.0000           288.0
                                               B cell       0.499         0.449    -0.050      0.050 -0.285 0.0000            75.0
                      CD4-positive, alpha-beta T cell       0.297         0.338     0.041      0.041  0.264 0.0000           791.0
                      pre-conventional dendritic cell       0.708         0.673    -0.035      0.035  0.009 0.8039            24.0
                                               T cell 

In [10]:
# ── Filter for significant monotonic trends (|τ| > 0.4 and p < 0.05) ──────
sig = df_scan[(df_scan['p_tau'] < 0.05) & (df_scan['tau'].abs() > 0.4)]
print(f"Cell types with significant monotonic F1 trend (|τ| > 0.4, p < 0.05): {len(sig)}")
print()
print(sig[['cell_type', 'f1_at_0pct', 'f1_at_100pct', 'delta_f1', 'tau', 'p_tau', 'median_support']].to_string(index=False))

Cell types with significant monotonic F1 trend (|τ| > 0.4, p < 0.05): 3

                                      cell_type  f1_at_0pct  f1_at_100pct  delta_f1    tau  p_tau  median_support
                                       platelet       0.991         0.968    -0.022 -0.597    0.0           181.0
effector memory CD4-positive, alpha-beta T cell       0.743         0.761     0.018  0.401    0.0           672.0
 central memory CD4-positive, alpha-beta T cell       0.804         0.818     0.014  0.426    0.0          2555.0


## Summary

| Cell type | Support | Δ F1 | Kendall τ | Notes |
|---|---|---|---|---|
| innate lymphoid cell | 16 | +0.132 | +0.709* | ILC_XCL1hi dominant subtype |
| double negative T reg | 4 | +0.140 | +0.491* | Very noisy; only 4 cells |
| pre-conventional DC | 24 | −0.035 | +0.127 (n.s.) | No significant monotonic trend |
| natural killer cell | 288 | +0.074 | +0.514* | Main finding — see NK notebook |
| B cell | 75 | −0.050 | −0.587* | Main finding — see B cell notebook |

(*) p < 0.05

**Interpretation:** Large Δ F1 with very small support (ILC, dnT) should be treated
with caution — a single annotation error flips F1 substantially. Only NK and B cells
have sufficient support (n > 50) for reliable trend estimation.
Pre-conv DC Δ F1 = −0.035 is not supported by a significant monotonic trend (τ = 0.127, p = 0.648).